In [1]:
import openmc
import os
import numpy as np
import re
from _Aplha_n_methods import Alpha_N_calc as an

np.set_printoptions(precision=3)
#import openmc.data
os.system('rm materials.xml')
openmc.reset_auto_ids()

def shell_vol_calc(outer_radius,inner_radius):
    volume=4/3*np.pi*(outer_radius**3-inner_radius**3)
    return volume
def sphere_cap_vol(r,h):
    return 1/3*np.pi*h**2*(3*r-h)
def weight_to_atompercent(nuclides,weights):
    #Takes array of weights of different isotopes and returns a atom percent for the nuclide
    result = []
    nr_of_atoms = []
    
    for nuc,weight in zip(nuclides,weights):
        atomic_mass = openmc.data.atomic_mass(nuc)#*1.66053906892e-27
        nr_of_atoms.append(weight/atomic_mass)

    total_nr_of_atoms = sum(nr_of_atoms)
    
    for atom in nr_of_atoms:
        result.append(atom/total_nr_of_atoms)
    return np.array(result)

def ao_to_wo(mat):
    wos = {}
    densities = mat.get_nuclide_atom_densities() #return atom densities in atom/b-cm (barn cm)
    for nuc_name,density in densities.items():
        mass = openmc.data.atomic_mass(nuc_name)
        wos[nuc_name]=mass*density/mat.density
    return wos
def atom_to_mol_ORNL(nuc):
    if nuc == 'Li':
        return 2
    elif nuc == 'Be':
        return 3
    elif nuc in ['Zr','Th','U']:
        return 5
    else:
        print(f'atom_to_mol_ORNL has not {nuc} defined')

def norm_array(array):
    return np.array(array)/sum(array)
inner_fuel_r = 0
outer_fuel_r = 55.28815478052896
inner_moderator_height = 84

outer_moderator_r = 87
blanket_or = 100


blanket_volume = shell_vol_calc(blanket_or,outer_moderator_r)
moderator_volume = shell_vol_calc(inner_fuel_r,0)-sphere_cap_vol(inner_fuel_r,inner_moderator_height)\
    +shell_vol_calc(outer_moderator_r,outer_fuel_r)
fuel_volume = shell_vol_calc(outer_fuel_r,inner_fuel_r)

print('fuel_volume',fuel_volume)

# Helper: make Li7-enriched lithium fluoride
LiF = openmc.Material(name='LiF')
LiF.add_elements_from_formula(formula='LiF',percent_type='ao',enrichment=99.9926,
                                 enrichment_target='Li7',enrichment_type='ao')

LiF_CA = openmc.Material(name='LiF_CA') #LiF with pure Li7
LiF_CA.add_elements_from_formula(formula='LiF',percent_type='ao',enrichment=99.999,
                                 enrichment_target='Li7',enrichment_type='ao')

BeF2 = openmc.Material(name='BeF2')
BeF2.add_elements_from_formula(formula='BeF2')

ZrF4 = openmc.Material(name='ZrF4')
ZrF4.add_elements_from_formula('ZrF4')  # 1 Zr : 4 F

ThF4 = openmc.Material(name='ThF4')
ThF4.add_elements_from_formula('ThF4')  # 1 Th : 4 F

UF4_LEU = openmc.Material(name='UF4_LEU')
UF4_LEU.add_elements_from_formula('UF4',percent_type='ao',enrichment=4.95
                                 ,enrichment_type='wo')
    #UF4_LEUeneriched to 4.95 % (LEU) and ao means by atom percent

UF4_A = openmc.Material(name='UF4_A')
#Using mass in kg from ORNL-TM-0611 p.7 to find atom percent 
atom_percent_U = weight_to_atompercent(['U234','U235','U236','U238'],[0.3,27,0.3,1.5])
atom_percent_UF4 = atom_percent_U*0.2
print("ao for UF4 in fuel A",atom_percent_UF4)
UF4_A.add_nuclide('U234',atom_percent_UF4[0],percent_type='ao')
UF4_A.add_nuclide('U235',atom_percent_UF4[1],percent_type='ao')
UF4_A.add_nuclide('U236',atom_percent_UF4[2],percent_type='ao')
UF4_A.add_nuclide('U238',atom_percent_UF4[3],percent_type='ao')
UF4_A.add_element('F',0.8,percent_type='ao')

UF4_B = openmc.Material(name='UF4_B')
#Using mass in kg from ORNL-TM-0611 p.7 to find atom percent 
atom_percent_U = weight_to_atompercent(['U234','U235','U236','U238'],[0.2,16.5,0.2,0.9])
atom_percent_UF4 = atom_percent_U*0.2
print("ao for UF4 in fuel B",atom_percent_UF4)
UF4_B.add_nuclide('U234',atom_percent_UF4[0])
UF4_B.add_nuclide('U235',atom_percent_UF4[1])
UF4_B.add_nuclide('U236',atom_percent_UF4[2])
UF4_B.add_nuclide('U238',atom_percent_UF4[3])
UF4_B.add_element('F',0.8)

UF4_C = openmc.Material(name='UF4_C')
#Using mass in kg from ORNL-TM-0611 p.7 to find atom percent 
atom_percent_U = weight_to_atompercent(['U234','U235','U236','U238'],[0.2,26.4,0.2,47.5])
atom_percent_UF4 = atom_percent_U*0.2
print("ao for UF4 in fuel C",atom_percent_UF4)
UF4_C.add_nuclide('U234',atom_percent_UF4[0])
UF4_C.add_nuclide('U235',atom_percent_UF4[1])
UF4_C.add_nuclide('U236',atom_percent_UF4[2])
UF4_C.add_nuclide('U238',atom_percent_UF4[3])
UF4_C.add_element('F',0.8)



#Define Copenhagen Atomic (CA) fuel salt
fuel_CA_1 = openmc.Material.mix_materials([LiF_CA,UF4_LEU,ThF4],[0.73,0.23,0.04],'ao',name="Fuel_CA_1") #from CA_1
fuel_CA_1.volume = fuel_volume
fuel_CA_1.depletable = True


fuel_CA_2 = openmc.Material.mix_materials([LiF_CA,UF4_LEU],[0.73,0.27],'ao',name="Fuel_CA_2") #from CA_2 table 1
fuel_CA_2.set_density('g/cm3',4.905)
fuel_CA_2.volume = fuel_volume #245700 in cm^3 from CA-2 table 2
fuel_CA_2.temperature = 626.8 + 273.15 # in Kelvin
fuel_CA_2.depletable = True

#Define Copenhagen Atomic blanket salt
Blanket_CA = openmc.Material.mix_materials([LiF,ThF4],[0.70,0.30],'ao',name="Blanket_CA")
#from CA-2 table 1
Blanket_CA.set_density('g/cm3',4.626)
Blanket_CA.volume = blanket_volume # in cm^3 from CA-2
Blanket_CA.temperature = 626.8 + 273.15 # in Kelvin
Blanket_CA.depletable = True


#lb/ft3 to g/cm3
lbft_to_gcm = 1.6018463374/100
# Define Fuel A
fuel_A = openmc.Material.mix_materials([LiF,BeF2,ZrF4,ThF4,UF4_A]
                                       ,norm_array([0.69987*2,0.237*3,0.05*5,0.01*5,0.00313*5])
                                       ,'ao',name='Fuel_A')
fuel_A.set_density('g/cm3',144.5*lbft_to_gcm)
fuel_A.volume = fuel_volume
fuel_A.depletable = True

# Fuel B
fuel_B = openmc.Material.mix_materials([LiF,BeF2,ZrF4,UF4_B]
                                       ,norm_array([0.66811*2,0.29*3,0.04*5,0.00189*5])
                                       ,'ao',name='Fuel_B')
fuel_B.set_density('g/cm3',134.5*lbft_to_gcm)
fuel_B.volume = fuel_volume
fuel_B.depletable = True

# Fuel C
fuel_C = openmc.Material.mix_materials([LiF,BeF2,ZrF4,UF4_C]
                                       ,norm_array([0.64969*2,0.292*3,0.05*5,0.00831*5])
                                       ,'ao',name='Fuel_C')
fuel_C.set_density('g/cm3', 142.7*lbft_to_gcm)
fuel_C.volume = fuel_volume
fuel_C.depletable = True

#Definition of air
N2 = openmc.Material(name='N2')
N2.add_elements_from_formula(formula='N2')

O2 = openmc.Material(name='O2')
O2.add_elements_from_formula(formula='O2')

Ar = openmc.Material(name='Ar')
Ar.add_elements_from_formula(formula='Ar')

CO2 = openmc.Material(name='CO2') #add_elements adds C0 so needed to be done "by hand"
#CO2.add_nuclide('C12', 1.0)
#CO2.add_nuclide('O16', 2.0)
CO2.add_elements_from_formula(formula='CO2') 

#With CO2
#air = openmc.Material.mix_materials([N2,O2,Ar,CO2],[0.7808,0.2095,0.0093,0.0004],'vo',name='Air')
#without CO2

air = openmc.Material.mix_materials([N2,O2,Ar],[0.7811,0.2096,0.0093],'vo',name='Air')



# Water 
water = openmc.Material(name="h2o")
water.add_nuclide('H1', 2.0)
water.add_nuclide('O16', 1.0)
water.set_density('g/cm3', 0.75)

water.add_s_alpha_beta('c_H_in_H2O')

# heavy_water 
heavy_water = openmc.Material(name="d2o")
heavy_water.add_nuclide('H2', 2.0)
heavy_water.add_nuclide('O16', 1.0)

heavy_water.set_density('g/cm3', 1.104) #from CA 2 table 1
heavy_water.volume = moderator_volume #2866100 in cm^3
heavy_water.temperature = 26.8 + 273.15 #in Kelvin
heavy_water.depletable = True

heavy_water.add_s_alpha_beta('c_D_in_D2O')


#UO2
uo2 = openmc.Material(name= "uo2",temperature=1200) #Available temperatures are 250, 294, 600, 900, 1200, 2500 K
# Add nuclides to uo2
uo2.add_nuclide('U235', 0.30)
uo2.add_nuclide('U238', 0.70)
uo2.add_nuclide('O16', 2.0)
uo2.set_density('g/cm3', 10.5)
uo2.volume = 1.0e4 
uo2.depletable = True


# Collect and export
materials = openmc.Materials([fuel_A, fuel_B, fuel_C, fuel_CA_1,fuel_CA_2, Blanket_CA, heavy_water, water,air, uo2])
materials.export_to_xml()
print(fuel_C)
print("✅ materials.xml has been created")

fuel_volume 707921.1647499999
ao for UF4 in fuel A [0.002 0.186 0.002 0.01 ]
ao for UF4 in fuel B [0.002 0.186 0.002 0.01 ]
ao for UF4 in fuel C [0.001 0.072 0.001 0.127]
Material
	ID             =	15
	Name           =	Fuel_C
	Temperature    =	None
	Density        =	2.2858347234698 [g/cm3]
	Volume         =	707921.1647499999 [cm^3]
	Depletable     =	True
	S(a,b) Tables  
	Nuclides       
	Li6            =	1.948861945819785e-05 [ao]
	Li7            =	0.2633402337885551 [ao]
	F19            =	0.594637869740933 [ao]
	Be9            =	0.11836574203564754 [ao]
	Zr90           =	0.010427940800914495 [ao]
	Zr91           =	0.002274081550753365 [ao]
	Zr92           =	0.0034759802669714988 [ao]
	Zr94           =	0.003522596911951292 [ao]
	Zr96           =	0.0005675069823626938 [ao]
	U234           =	9.180444797269919e-06 [ao]
	U235           =	0.0012066476488254612 [ao]
	U236           =	9.102479914025151e-06 [ao]
	U238           =	0.002143628728916089 [ao]

✅ materials.xml has been created


In [2]:
def check_fuel_from_ORNL(fuel):
    print(fuel.name)
    nuclide_lib = {}
    total = 0
    for nuc,dens in fuel.get_nuclide_densities().items():
        symbol,Z,A = an.isolate_atomic_symbol(nuc)
        if symbol != 'F':
            #print(nuc,dens)
            mol = dens.percent
            total += mol
        if symbol in nuclide_lib:
            nuclide_lib[symbol]+=mol
        else:
            nuclide_lib[symbol]=mol
    
    for nuc in nuclide_lib:
        if nuc != 'F':
            print(f'{nuc}: {nuclide_lib[nuc]/total*100:.10g}')

check_fuel_from_ORNL(fuel_C)


Fuel_C
Li: 64.969
Be: 29.2
Zr: 5
U: 0.831


In [3]:
inner_fuel_r = 0
outer_fuel_r = 50
V = 707921.16475
r = np.cbrt(3/(4*np.pi)*V)
outer_fuel_r = r
fuel_volume = shell_vol_calc(outer_fuel_r,inner_fuel_r)

print(r)
print(fuel_volume-707921.16475)

55.28815478052896
-1.1641532182693481e-10


In [4]:
#print(fuel_A.get_nuclide_densities().items())
for nuc,dens in fuel_A.get_nuclide_densities().items():
    print(nuc,dens.percent)

Li6 2.134462308203254e-05
Li7 0.28841950783674514
F19 0.5878651000045336
Be9 0.09767597129892558
Zr90 0.010602170302383378
Zr91 0.002312076788974568
Zr92 0.0035340567674611266
Zr94 0.0035814522809606056
Zr96 0.0005769888599936533
Th230 8.242697999909335e-07
Th232 0.004120524730154677
U234 1.3364473321824988e-05
U235 0.001197670008050427
U236 1.3250975596477986e-05
U238 6.569678001708054e-05


In [5]:
def deviation_calc(x,y):
    return (y-x)/x*100

print('Fuel A')
for a,b in zip([0.2977497591,26.79747832,1.488748796],[0.3,27,1.5]):
    print(f'{deviation_calc(b,a):.5g} %')
print('Fuel B')
for a,b in zip([0.1989641319,16.41454088,0.8953385936],[0.2,16.5,0.9]):
    print(f'{deviation_calc(b,a):.5g} %')
print('Fuel C')
for a,b in zip([0.2062339039,27.22287532,48.98055218],[0.2,26.4,47.5]):
    print(f'{deviation_calc(b,a):.5g} %')

Fuel A
-0.75008 %
-0.75008 %
-0.75008 %
Fuel B
-0.51793 %
-0.51793 %
-0.51793 %
Fuel C
3.117 %
3.117 %
3.117 %
